In [4]:
!pip install duckdb pandas matplotlib seaborn plotly -q

import pandas as pd
import duckdb
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("Setup terminé ! DuckDB est prêt à l'emploi.")
print(f"Version DuckDB : {duckdb.__version__}")
print(f"Version Pandas : {pd.__version__}")

Setup terminé ! DuckDB est prêt à l'emploi.
Version DuckDB : 1.4.4
Version Pandas : 2.3.3


In [5]:
from pathlib import Path

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR



WindowsPath('C:/Users/MGI/Desktop/Recommendation_system/data/processed')

In [6]:
import duckdb

con = duckdb.connect()

csv_path = PROCESSED_DIR / "final_table.csv"

df = con.execute(f"""
    SELECT *
    FROM read_csv('{csv_path.as_posix()}', header=true, encoding='utf-8')
    LIMIT 5
""").df()

con.close()
df


IOException: IO Error: No files found that match the pattern "C:/Users/MGI/Desktop/Recommendation_system/data/processed/final_table.csv"

LINE 3:     FROM read_csv('C:/Users/MGI/Desktop/Recommendation_system/data...
                 ^

In [ ]:
df.describe

<bound method NDFrame.describe of                            order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   
4  a4591c265e18cb1dcee52889e2d8acc3  503740e9ca751ccdda7ba28e9ab8f608   

  order_status order_purchase_timestamp   order_approved_at  \
0    delivered      2017-10-02 10:56:33 2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37 2018-07-26 03:24:27   
2    delivered      2018-08-08 08:38:49 2018-08-08 08:55:23   
3    delivered      2018-02-13 21:18:39 2018-02-13 22:20:29   
4    delivered      2017-07-09 21:57:05 2017-07-09 22:10:13   

  order_delivered_carrier_date order_delivered_customer_date  \
0          2017-10-04 19:55:00           2017-10-10 21:25:13   
1          2018-07-2

In [ ]:
df.isnull().sum()

order_id                             0
customer_id                          0
order_status                         0
order_purchase_timestamp             0
order_approved_at                    0
order_delivered_carrier_date         0
order_delivered_customer_date        0
order_estimated_delivery_date        0
med_approve_delay                    0
med_carrier_delay                    0
med_customer_delay                   0
order_approved_at_imp                0
order_delivered_carrier_date_imp     0
order_delivered_customer_date_imp    0
customer_unique_id                   0
customer_zip_code_prefix             0
customer_city                        0
customer_state                       0
nb_items                             0
nb_distinct_products                 0
nb_distinct_sellers                  0
items_price_sum                      0
freight_sum                          0
payment_value_sum                    0
nb_payments                          0
max_installments         

Parfait, notre base de données a été bien traitrée et nous permet enfin d'appliquer les algorithmes.

In [ ]:
import duckdb

con = duckdb.connect()

con.execute(f"""
    CREATE OR REPLACE TEMP TABLE final_table AS
    SELECT *
    FROM read_csv_auto('{csv_path.as_posix()}');
""")



In [ ]:
con.execute("""
SELECT
  (SELECT COUNT(*) FROM final_table) AS total_rows,
  (SELECT COUNT(*) FROM (SELECT DISTINCT * FROM final_table) t) AS distinct_rows,
  (SELECT COUNT(*) FROM final_table)
  - (SELECT COUNT(*) FROM (SELECT DISTINCT * FROM final_table) t) AS duplicate_rows
""").df()


,total_rows,distinct_rows,duplicate_rows
0,99441,99441,0


De plus, nous n'avons aucun doublon. Ce resultat temoigne de la qualité de nos données.

🚀 Allons-y, avançons avec ambition et détermination vers la modélisation, là où les idées deviennent des résultats concrets ! 🚀